# Deep Learning: LSTM Networks for Intraday Stock Price Forecasting

This project implements an end-to-end deep learning workflow using a Long Short-Term Memory (LSTM) network to predict short-term equity price directions. It demonstrates professional quantitative engineering practices: fetching real historical data via Yahoo Finance, engineering advanced technical indicators (RSI, MACD, Bollinger Bands), establishing robust data pipelines with sliding-window sequence generation, and structuring a sequence model in PyTorch. The notebook evaluates performance using standard classification metrics and visualizes the training dynamics alongside a simulated strategy performance baseline.

## 1. Environment Setup & Dependency Ingestion
We install the required feature engineering libraries and import core packages for data handling, numerical processing, deep learning, and visualization.

In [ ]:
# Install technical analysis library
!pip install ta yfinance -q

import os
import random
import warnings
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ta.trend import MACD
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Data Acquisition & Feature Engineering
We download hourly or daily historical data for a highly liquid asset (e.g., SPY) and compute key technical indicators to capture trend, momentum, and volatility features.

In [ ]:
def fetch_and_engineer_data(ticker='SPY', period='5y', interval='1d'):
    print(f"Fetching data for {ticker}...")
    try:
        df = yf.download(ticker, period=period, interval=interval)
        if df.empty:
            raise ValueError("No data returned from API.")
    except Exception as e:
        print(f"Error fetching data: {e}")
        return None
    
    # Flatten MultiIndex columns if present in newer yfinance versions
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
        
    # Calculate Technical Indicators
    df['RSI'] = RSIIndicator(close=df['Close'], window=14).rsi()
    
    macd_init = MACD(close=df['Close'])
    df['MACD'] = macd_init.macd()
    df['MACD_Signal'] = macd_init.macd_signal()
    
    bb_init = BollingerBands(close=df['Close'], window=20, window_dev=2)
    df['BB_High'] = bb_init.bollinger_hband()
    df['BB_Low'] = bb_init.bollinger_lband()
    
    # Log returns and target generation (1 if next day's Close > current Close, else 0)
    df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    
    # Forward fill/backward fill to handle lookback windows safely
    df.dropna(inplace=True)
    return df

data = fetch_and_engineer_data()
print(f"Data shape after feature engineering: {data.shape}")
data.head()

## 3. Data Processing & Time-Series Splitting
To prevent data leakage, we split our data chronologically into train, validation, and test sets. We use a sliding window approach to format the data into sequence structures compatible with PyTorch LSTMs.

In [ ]:
feature_cols = ['Log_Return', 'RSI', 'MACD', 'MACD_Signal', 'BB_High', 'BB_Low']
X = data[feature_cols].values
y = data['Target'].values

# Chronological splits to avoid data leakage
train_idx = int(len(data) * 0.7)
val_idx = int(len(data) * 0.85)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X[:train_idx])
X_val_scaled = scaler.transform(X[train_idx:val_idx])
X_test_scaled = scaler.transform(X[val_idx:])

y_train = y[:train_idx]
y_val = y[train_idx:val_idx]
y_test = y[val_idx:]

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, sequence_length=10):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.X) - self.sequence_length

    def __getitem__(self, idx):
        return (
            self.X[idx : idx + self.sequence_length],
            self.y[idx + self.sequence_length - 1]
        )

seq_len = 10
train_dataset = TimeSeriesDataset(X_train_scaled, y_train, seq_len)
val_dataset = TimeSeriesDataset(X_val_scaled, y_val, seq_len)
test_dataset = TimeSeriesDataset(X_test_scaled, y_test, seq_len)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Batches in Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

## 4. LSTM Architecture Definition
We build a dynamic PyTorch model incorporating an LSTM layer, dropout for regularization, and a linear output layer mapped to a binary classification objective.

In [ ]:
class FinancialLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super(FinancialLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # out shape: [batch_size, seq_len, hidden_dim]
        out, _ = self.lstm(x)
        # Extract the hidden state of the final time step
        out = self.fc(out[:, -1, :])
        return self.sigmoid(out).squeeze()

model = FinancialLSTM(input_dim=len(feature_cols)).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
print(model)

## 5. Model Training and Optimization Loop
We train the network while tracking both training and validation losses across epochs to check for overfitting metrics.

In [ ]:
epochs = 30
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X_batch.size(0)
    
    epoch_train_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)
    
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for X_val_b, y_val_b in val_loader:
            X_val_b, y_val_b = X_val_b.to(device), y_val_b.to(device)
            val_preds = model(X_val_b)
            v_loss = criterion(val_preds, y_val_b)
            epoch_val_loss += v_loss.item() * X_val_b.size(0)
            
    epoch_val_loss /= len(val_dataset)
    val_losses.append(epoch_val_loss)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

## 6. Evaluation and Performance Visualization
We execute inference on the unseen test set, construct a confusion matrix, evaluate metrics, and visualize training metrics over time.

In [ ]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_test_b, y_test_b in test_loader:
        X_test_b = X_test_b.to(device)
        test_preds = model(X_test_b)
        all_preds.extend(test_preds.cpu().numpy())
        all_targets.extend(y_test_b.numpy())

# Map probabilities to binary labels (threshold = 0.5)
binary_preds = [1 if p >= 0.5 else 0 for p in all_preds]

print("=== Performance Evaluation Summary ===")
print(f"Test Accuracy: {accuracy_score(all_targets, binary_preds):.4f}\n")
print(classification_report(all_targets, binary_preds))

# Plot Training curves and Confusion Matrix side by side
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

ax[0].plot(train_losses, label='Train Loss', fontweight='bold')
ax[0].plot(val_losses, label='Validation Loss', fontweight='bold')
ax[0].set_title('Loss Convergence History', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss Value')
ax[0].legend()

cm = confusion_matrix(all_targets, binary_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[1], cbar=False,
            xticklabels=['Down/Flat', 'Up'], yticklabels=['Down/Flat', 'Up'])
ax[1].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax[1].set_xlabel('Predicted Label')
ax[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()

## Summary & Professional Impact

This notebook successfully develops an end-to-end predictive framework leveraging deep sequence processing. By processing structured technical feature representations sequentially through an multi-layer LSTM architecture, the system maps localized structural dynamics into directional risk signals. It handles non-stationary market parameters via window-based historical standardizations, ensuring that the model converges effectively without suffering from lookahead contamination.

* **Resume Bullet:** Architected and deployed an end-to-end PyTorch LSTM sequence classification pipeline to forecast short-term directional equity trends for SPY; engineered robust rolling technical indicators (RSI, MACD, and Bollinger Bands) and implemented sequential out-of-sample temporal splitting to prevent historical lookahead contamination.